# 01 Python の for ループと NumPy のベクトル演算

## 1. 研究目的（RQ1・RQ2）

- **同じ数学的計算**を、Python の list と for ループ（`cpu_basics.py`）と、NumPy の配列演算（`numpy_benchmark.py`）で実行し、**結果が一致すること**と**実行時間**を比べる。
- **問題サイズ**（要素数 $n$）によって、どちらが速いか・どれくらい速いか（高速化率）が変わるかを調べる。

$$ \text{Speedup} = \frac{\text{Python の実行時間}}{\text{NumPy の実行時間}} $$

Speedup が 1 より大きければ NumPy の方が速く、1 より小さければ Python の方が速いことを表します。

## 2. 理論的背景

| Task | 計算 | Python（`cpu_basics.py`） | NumPy（`numpy_benchmark.py`） | 計算量 |
| --- | --- | --- | --- | --- |
| A 要素ごとの演算 | $y_i = a_i b_i + c_i$ | for ループ | `a * b + c` | $O(n)$ |
| B 総和 | $s = \sum_i a_i$ | for ループ | `np.sum(a)` | $O(n)$ |
| C 行列積 | $C_{ij} = \sum_k A_{ik} B_{kj}$ | 3 重 for ループ | `A @ B` | $O(n^3)$（$n \times n$） |
| D 条件分岐 | $y_i = x_i^2$（$x_i > 0$）、$0$（それ以外） | if 文 | `np.where(x > 0, x * x, 0)` | $O(n)$ |

### なぜ NumPy は速いことが多いのか

- **list**：要素 1 つ 1 つが別々の Python オブジェクトで、メモリ上にばらばらに置かれる。for ループでは、Python のインタプリタが 1 要素ごとに「型を調べる → 計算する」を繰り返す。
- **NumPy 配列**：同じ型（dtype）の値が**連続したメモリ**に並ぶ。ループは C 言語の中で回り、CPU の **SIMD 命令**（1 命令で複数の値を計算）や、行列積では最適化された **BLAS** ライブラリを使える。

### ただし

- NumPy の関数呼び出しには**固定のコスト**があるので、要素数が少ないと Python の方が速いこともあります。
- ベクトル化しても**計算量のオーダーは変わりません**（$O(n)$ は $O(n)$ のまま）。変わるのは定数倍（1 要素あたりの時間）です。
- `np.where` は**両方の分岐**（`x * x` と `0`）を全要素について計算します。

## 3. 実験環境（最初に必ず実行）

このセルは、E2 リポジトリの `.py`（`cpu_basics.py`、`numpy_benchmark.py` など）を import できるように、
**作業ディレクトリ（E2 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E2 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E2 フォルダを ZIP にして `/content` にアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/`（教科書のコード）は**公開リポジトリには含めない**約束です。
> `textbook/` がないと `algorithm_benchmark.py` の import が失敗します（Notebook 04・05 の一部）。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e2-computing-performance"   # リポジトリの中の E2 フォルダ（ルートが E2 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e2_root(p):
    """cpu_basics.py と benchmark_utils.py があるフォルダを E2 のルートとみなす"""
    return (Path(p) / "cpu_basics.py").exists() and (Path(p) / "benchmark_utils.py").exists()


def find_e2_root():
    # 1) ローカル：notebooks/ の中、または E2 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e2_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e2_upload")
    for hit in glob.glob("/content/e2_upload/**/cpu_basics.py", recursive=True):
        if is_e2_root(Path(hit).parent):
            return Path(hit).parent
    return None


E2_ROOT = find_e2_root()
if E2_ROOT is None:
    raise RuntimeError("E2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E2_ROOT)
if str(E2_ROOT) not in sys.path:
    sys.path.insert(0, str(E2_ROOT))
print("E2_ROOT:", E2_ROOT)
if not (E2_ROOT / "textbook" / "bsearch.py").exists():
    print("⚠️ textbook/ がありません。algorithm_benchmark.py の import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は「実装（ライブラリ）」ごとに固定する。デバイス（CPU / GPU / TPU）は線の種類と記号で区別する
COLORS = {"Python": "#eb6834", "NumPy": "#2a78d6", "PyTorch": "#1baf7a", "JAX": "#6250d6",
          "TensorFlow": "#eda100", "neutral": "#8a8985"}
DEVICE_STYLE = {"CPU": ("-", "o"), "GPU": ("--", "s"), "TPU": ("-.", "^")}

In [ ]:
import platform

def show_environment():
    """Python・ライブラリのバージョン、CPU、GPU・TPU の有無を表示する（測定はしない）"""
    info = {"Python": platform.python_version(), "Platform": platform.platform()}
    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as f:
            cpu = next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        pass
    info["CPU"] = f"{cpu} ({os.cpu_count()} logical cores)"
    try:
        import numpy
        info["NumPy"] = numpy.__version__
    except ImportError:
        info["NumPy"] = "not installed"
    try:
        import torch
        info["PyTorch"] = torch.__version__
        info["CUDA GPU"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available"
    except ImportError:
        info["PyTorch"] = "not installed"
        info["CUDA GPU"] = "unknown (PyTorch not installed)"
    try:
        import jax
        info["JAX"] = jax.__version__
        info["JAX devices"] = ", ".join(f"{d.platform}:{d.device_kind}" for d in jax.devices())
        info["TPU"] = "available" if any(d.platform == "tpu" for d in jax.devices()) else "not available"
    except ImportError:
        info["JAX"] = "not installed"
        info["TPU"] = "unknown (JAX not installed)"
    except Exception as e:   # TPU の初期化に失敗した場合など
        info["JAX"] = info.get("JAX", "installed") + f" (device error: {e})"
        info["TPU"] = "not available"
    for k, v in info.items():
        print(f"{k:12}: {v}")
    return info

ENV = show_environment()

## 4. 使用するコード

| ファイル | 使う関数 |
| --- | --- |
| `cpu_basics.py` | `elementwise`、`total`、`matmul`、`conditional_square`、`make_vector`、`make_matrix` |
| `numpy_benchmark.py` | `np_*` 関数、`make_inputs`（同じ値の list と配列を作る）、`compare`（1 条件の比較）、`results_match`、安全上限 `PYTHON_MAX_N` |
| `benchmark_utils.py` | `measure`（ウォームアップ＋複数回測定の中央値・平均・標準偏差）、`format_table` |

### 計測の約束

- データの生成（list・配列の作成）は**計算時間に含めない**（`data_gen_s` に別に記録）
- ウォームアップのあと複数回測り、**中央値**を使う。乱数シード・dtype・問題サイズ・反復回数を記録する
- 結果の正しさは `np.allclose`（浮動小数点の丸め誤差を許す比較）で確認する

In [ ]:
import sys
import numpy as np

import cpu_basics as cb
import numpy_benchmark as nb
from benchmark_utils import measure, format_table

SEED = 0
REPEATS = 5
print("Python ループの安全上限:", nb.PYTHON_MAX_N)

## 5. 実行例

### 5-1. Python の list と NumPy 配列

In [ ]:
lst = [1.0, 2.0, 3.0, 4.0]
arr = np.array(lst)
print("list :", lst, type(lst))
print("array:", arr, type(arr), "dtype =", arr.dtype, "shape =", arr.shape)

# メモリの大きさの比較（list は「要素への参照」と各 float オブジェクトの両方が必要）
n = 100_000
big_list = cb.make_vector(n, seed=SEED)
big_arr = np.asarray(big_list)
list_bytes = sys.getsizeof(big_list) + sum(sys.getsizeof(v) for v in big_list)
print(f"n = {n:,}: list ≈ {list_bytes:,} bytes, NumPy float64 = {big_arr.nbytes:,} bytes, "
      f"float32 = {big_arr.astype(np.float32).nbytes:,} bytes")

### 5-2. 4 つの Task の結果が一致するか

小さな入力で、Python と NumPy の計算結果を並べて確認します。

In [ ]:
a, b, c = [1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [0.5, 0.5, 0.5]
x = [-2.0, -0.5, 0.0, 1.5, 3.0]
A, B = [[1.0, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]

checks = [
    ("A elementwise", cb.elementwise(a, b, c), nb.np_elementwise(np.array(a), np.array(b), np.array(c))),
    ("B total", cb.total(a), nb.np_total(np.array(a))),
    ("C matmul", cb.matmul(A, B), nb.np_matmul(np.array(A), np.array(B))),
    ("D conditional", cb.conditional_square(x), nb.np_conditional_square(np.array(x))),
]
for name, py_res, np_res in checks:
    ok = np.allclose(py_res, np_res)
    print(f"{name:14} Python: {py_res}\n{'':14} NumPy : {np_res.tolist()}   match: {ok}")
    assert ok
assert cb.matmul(A, B) == [[19.0, 22.0], [43.0, 50.0]]

### 5-3. 参考 Notebook の実験：for ループの総和 vs NumPy の総和

参考 Notebook（`A_cpu_gpu_tpu_benchmark/starter.ipynb`）の最初の実験を、E2 の計測方法で測り直します。

参考 Notebook からの変更点：
- `np.arange` による**データの生成を計算時間から外した**（参考 Notebook では生成と総和を一緒に測っていた）
- 1 回だけでなく、**ウォームアップのあと複数回測って中央値**を使う
- $N$ を 2,000,000 から 1,000,000 に下げた（Python ループの安全上限）

In [ ]:
N = 1_000_000   # 参考 Notebook は 2_000_000

def loop_sum():                      # 参考 Notebook と同じ for ループ（整数の和）
    total = 0
    for i in range(N):
        total += i
    return total

arr_N = np.arange(N, dtype=np.int64)  # データの生成は計算時間の外で行う
t_loop = measure(loop_sum, repeats=3, warmup=1)
t_np = measure(lambda: arr_N.sum(), repeats=REPEATS, warmup=1)
assert loop_sum() == int(arr_N.sum()) == N * (N - 1) // 2
print(f"for loop : median {t_loop['median']:.4f} s (mean {t_loop['mean']:.4f}, stdev {t_loop['stdev']:.4f})")
print(f"NumPy sum: median {t_np['median']:.6f} s")
print(f"speedup  : {t_loop['median'] / t_np['median']:.0f}x")

### 5-4. データサイズを変えて比較する（RQ2）

要素ごとの演算・総和・条件分岐は $n$ = 100 〜 1,000,000、行列積は $n \times n$ で $n$ = 10 〜 200 を測ります。
Python ループの安全上限を超える条件は**実行せず**、表では `-` になります。

In [ ]:
SIZES = [100, 1_000, 10_000, 100_000, 1_000_000]
MATMUL_SIZES = [10, 25, 50, 100, 200]

rows = []
for task in nb.TASKS:
    for n in (MATMUL_SIZES if task == "matmul" else SIZES):
        reps = 3 if n >= 100_000 or (task == "matmul" and n >= 100) else REPEATS
        rows.append(nb.compare(task, n, repeats=reps, seed=SEED))

print(format_table(rows, [("task", "task"), ("n", "n"), ("dtype", "dtype"), ("repeats", "repeats"),
                          ("data_gen_s", "data gen"), ("python_s", "Python"), ("numpy_s", "NumPy"),
                          ("speedup", "speedup"), ("match", "match")]))
assert all(r["match"] for r in rows if r["match"] is not None)

## 6. 結果の可視化

### 6-1. データサイズと実行時間（Python と NumPy）

両対数グラフでは、$O(n)$ の計算は傾き 1、$O(n^3)$ の計算は傾き 3 の直線に近くなります。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, task in zip(axes, nb.TASKS):
    sel = [r for r in rows if r["task"] == task]
    for impl, key in [("Python", "python_s"), ("NumPy", "numpy_s")]:
        pts = [(r["n"], r[key]) for r in sel if r[key] is not None]
        if pts:
            xs, ys = zip(*pts)
            ax.loglog(xs, ys, marker="o", ms=6, color=COLORS[impl], label=impl)
    ax.set_title(task); ax.set_xlabel("n (matrix: n x n)" if task == "matmul" else "n")
axes[0].set_ylabel("median time [s]"); axes[0].legend()
plt.tight_layout(); plt.show()

### 6-2. 高速化率（Speedup = Python / NumPy）

破線の 1 より上なら NumPy の方が速い。

In [ ]:
plt.figure(figsize=(8, 3.8))
markers = {"elementwise": "o", "total": "s", "conditional": "^", "matmul": "D"}
task_colors = {"elementwise": "#2a78d6", "total": "#eb6834", "conditional": "#1baf7a", "matmul": "#6250d6"}
for task in nb.TASKS:
    pts = [(r["n"], r["speedup"]) for r in rows if r["task"] == task and r["speedup"] is not None]
    if pts:
        xs, ys = zip(*pts)
        plt.loglog(xs, ys, marker=markers[task], ms=6, color=task_colors[task], label=task)
plt.axhline(1, color="black", ls="--", lw=1)
plt.xlabel("n (matmul: n x n)"); plt.ylabel("speedup (Python / NumPy)")
plt.title("speedup of NumPy over Python loops"); plt.legend()
plt.show()

## 7. 結果の解釈（読み方のヒント）

- **小さい $n$**：NumPy の呼び出しの固定コストが目立ち、speedup が小さい（1 未満になることもある）。
- **大きい $n$**：1 要素あたりの時間の差が効き、speedup が大きくなる。ただし、配列がキャッシュに入らなくなると NumPy 側も**メモリの読み書きの速さ**で頭打ちになり、speedup が下がることがある。
- **行列積**：speedup が特に大きくなりやすい。BLAS はデータを再利用する工夫（ブロック化）や複数コアを使うため。
- **条件分岐**：`np.where` は両方の分岐を計算するので、他の要素ごとの演算より speedup が小さくなることがある。
- **計算量**：両対数グラフの傾き（オーダー）は Python と NumPy でほぼ同じ。**ベクトル化は定数倍を改善するが、オーダーは変えない**。

## 8. 学生向け TODO

- [ ] **TODO 1**：データサイズ `SIZES` を変更する（安全上限 `nb.PYTHON_MAX_N` に注意）。
- [ ] **TODO 2**：配列の dtype を変更する（下のセル。`np.float32` で時間と誤差がどう変わるか）。
- [ ] **TODO 3**：行列サイズ `MATMUL_SIZES` を変更する。
- [ ] **TODO 4**：演算の種類を変更する（例：`a * b + c` を `np.sqrt(np.abs(a))` などに。`cpu_basics.py` にも同じ計算を追加する）。
- [ ] **TODO 5**：Task ごとの高速化率を比較し、差が出る理由を考える。
- [ ] **TODO 6**：list → 配列の変換時間（`data_gen_s` に含まれる）を計算時間に加えた場合の speedup を計算する。

In [ ]:
# TODO 2：dtype を変えて比べる
for dtype in [np.float64, np.float32]:
    r = nb.compare("elementwise", 100_000, repeats=3, seed=SEED, dtype=dtype)
    print(f"{r['dtype']:8} NumPy {r['numpy_s'] * 1e6:8.1f} µs   speedup {r['speedup']:6.1f}   match {r['match']}")
# TODO：total・matmul でも試し、結果の誤差（np.max(np.abs(...))）も記録しよう

## 9. 考察課題

1. 同じ $O(n)$ の計算なのに、Python と NumPy で実行時間が何倍も違うのはなぜですか。「インタプリタ」「メモリ配置」「SIMD」という言葉を使って説明してください。
2. speedup が最も大きかった Task と最も小さかった Task は何ですか。理由を考えてください。
3. NumPy が Python より遅くなる条件はありましたか。それはなぜですか。
4. 「NumPy を使えばアルゴリズムの計算量が改善する」という主張は正しいですか。グラフの傾きを根拠に説明してください。

## 10. 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第6章 ソート）
- 参考 Notebook：`ai-programming-2026-team-project-1-starter/A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License, Copyright (c) 2026 Daigo Takeuchi）
- NumPy ドキュメント <https://numpy.org/doc/>
- PyTorch ドキュメント：CUDA semantics（非同期実行と同期）<https://pytorch.org/docs/stable/notes/cuda.html>
- JAX ドキュメント：Asynchronous dispatch、Just-in-time compilation <https://docs.jax.dev/>

- E2 リポジトリの `README.md`